## Virtual threads — Java 21's big shift

Until Java 21, every Java thread was a **platform thread** — a thin wrapper around an OS thread. OS threads are expensive: about 1 MB of stack, kernel scheduling overhead, and only around ten thousand in flight before a machine struggles.

**Virtual threads** are a JVM-managed alternative. They're lightweight — kilobytes, not megabytes — and the JVM multiplexes many of them onto a small pool of platform threads called *carriers*. Blocking a virtual thread on I/O *parks* it, and the carrier moves on to another virtual thread.

```java
// one virtual thread per task — no pool tuning, no async callbacks
try (var exec = Executors.newVirtualThreadPerTaskExecutor()) {
    var futures = IntStream.range(0, 10_000)
        .mapToObj(i -> exec.submit(() -> {
            Thread.sleep(100);   // parks the virtual thread; frees the carrier
            return i * 2;
        }))
        .toList();
    int sum = 0;
    for (var f : futures) sum += f.get();
}
```

The practical effect: you can have **millions** of concurrent virtual threads waiting on the network. Code that used to need a non-blocking framework — Netty, Reactor, async/await — can be plain, blocking, one-thread-per-request code, and still scale.

Use virtual threads for **I/O-bound** work (HTTP servers, DB calls) — the whole reason they exist. Use platform threads for **CPU-bound** work (a virtual thread that never blocks just pins its carrier). And don't use `synchronized` on virtual-thread hot paths — it pins the thread to its carrier; use `ReentrantLock` instead.
